In [1]:
import platform
import sys
import numpy as np
import pandas as pd
import scipy
import sklearn
import matplotlib

print(f"Python       {sys.version.split()[0]} on {platform.system()}")
for lib in (np, pd, scipy, sklearn, matplotlib):
    print(f"{lib.__name__:<12} {lib.__version__}")

# Optional extras. The notebooks fall back gracefully if these are absent.
for name in ("seaborn", "statsmodels"):
    try:
        mod = __import__(name)
        print(f"{name:<12} {mod.__version__}")
    except ImportError:
        print(f"{name:<12} not installed (optional)")

Python       3.11.16 on Windows
numpy        2.4.6
pandas       3.0.5
scipy        1.17.1
sklearn      1.9.0
matplotlib   3.11.1
seaborn      0.13.2
statsmodels  0.15.0


In [2]:
from pathlib import Path
import matplotlib.pyplot as plt

ROOT = Path.cwd()
# data files may sit in ./data or right next to the notebook
DATA = next((p for p in (ROOT / "data", ROOT) if (p / "anggota.csv").exists()), ROOT / "data")
pd.set_option("display.width", 110)
pd.set_option("display.max_columns", 30)
print("data folder:", DATA)

data folder: c:\Users\ahmad\dev-resources\S2\data-science\project-pasca-uts\data


In [20]:
SENTINELS = ["", "NA", "N/A", "-", "null", "NULL", "unknown", "-999"]

rawDesa = pd.read_excel(
    DATA / "desa.xlsx",
    dtype={"id": "string"},             # preserve leading zeros
    na_values=SENTINELS,                # declare the folk sentinels
    keep_default_na=True,
    parse_dates=["dihitung_pada"],
    # dayfirst=True,                      # 03/04/2023 is 3 April
)
print("shape:", rawDesa.shape)
print(rawDesa.dtypes.to_string())
rawDesa.head(5)

shape: (79842, 33)
kode                               int64
nama                                 str
tipe                                 str
kecamatan                            str
kabupaten                            str
provinsi                             str
luas_desa                        float64
jumlah_persil                      int64
luas_persil                        int64
hak_milik_jumlah                   int64
hak_milik_luas                     int64
hgb_jumlah                         int64
hgb_luas                           int64
hgu_jumlah                         int64
hgu_luas                           int64
hak_pakai_jumlah                   int64
hak_pakai_luas                     int64
hak_pengelolaan_jumlah             int64
hak_pengelolaan_luas               int64
hak_wakaf_jumlah                   int64
hak_wakaf_luas                     int64
kosong_jumlah                      int64
kosong_luas                        int64
lainnya_jumlah                     int

,kode,nama,tipe,kecamatan,kabupaten,provinsi,luas_desa,jumlah_persil,luas_persil,hak_milik_jumlah,hak_milik_luas,hgb_jumlah,hgb_luas,hgu_jumlah,hgu_luas,...,hak_pengelolaan_luas,hak_wakaf_jumlah,hak_wakaf_luas,kosong_jumlah,kosong_luas,lainnya_jumlah,lainnya_luas,tahun_min,tahun_median,tahun_max,rdtr_luas,banjir_rendah,banjir_sedang,banjir_tinggi,dihitung_pada
0,1010101,KOTA BARU,Kelurahan,Kuta Alam,Banda Aceh,Aceh,593698.7,313,679003,219,95859,1,6125,0,0,...,0,0,0,46,181137,0,0,1972.0,2006.0,2026.0,593686.4,22987.2,130725.3,439986.2,2026-10-17 15:12:38
1,1010102,BANDAR BARU,Kelurahan,Kuta Alam,Banda Aceh,Aceh,1139905.6,797,1016060,668,294883,20,8270,0,0,...,0,5,1299,43,84498,0,0,1972.0,2004.0,2026.0,1139905.6,0.0,815590.7,324315.3,2026-10-17 15:12:38
2,1010103,BEURAWE,Kelurahan,Kuta Alam,Banda Aceh,Aceh,799736.9,1601,737670,1455,455608,19,27576,0,0,...,0,2,472,93,190819,0,0,1974.0,2012.0,2026.0,799736.9,0.0,350871.9,448864.9,2026-10-17 15:12:38
3,1010104,KUTA ALAM,Kelurahan,Kuta Alam,Banda Aceh,Aceh,511125.6,576,553700,417,86916,38,25835,0,0,...,2572,2,199,76,132973,0,0,1964.0,2003.0,2026.0,511112.5,0.0,501677.5,9448.1,2026-10-17 15:12:38
4,1010105,KEURAMAT,Kelurahan,Kuta Alam,Banda Aceh,Aceh,326105.0,932,336401,792,174164,18,12373,0,0,...,0,3,1309,79,52816,0,0,1974.0,2001.0,2026.0,326105.0,0.0,326105.0,0.0,2026-10-17 15:12:38


In [17]:
import pandas as pd

def profile(df):
    out = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_missing": df.isna().sum(),
        "pct_missing": (df.isna().mean() * 100).round(1),
        "n_unique": df.nunique(dropna=True),
        "example": [
            df[c].dropna().sample(n=min(2, df[c].notna().sum())).tolist() 
            if df[c].notna().any() else []
            for c in df.columns
        ],
    })
    return out.sort_values("pct_missing", ascending=False)

In [19]:
profile(rawDesa)

,dtype,n_missing,pct_missing,n_unique,example
tahun_max,float64,2670,3.3,61,"[2025.0, 2018.0]"
tahun_median,float64,2670,3.3,62,"[2021.0, 2019.0]"
tahun_min,float64,2670,3.3,67,"[2012.0, 2011.0]"
kode,int64,0,0.0,79842,"[10080302, 2040316]"
nama,str,1,0.0,63428,"[Larete, Dorolamo]"
provinsi,str,0,0.0,34,"[Nusa Tenggara Barat, Nusa Tenggara Timur]"
tipe,str,0,0.0,2,"[Desa, Kelurahan]"
kecamatan,str,1,0.0,6732,"[Dumoga Tengah, Sayung]"
kabupaten,str,0,0.0,487,"[Malaka, GORONTALO]"
hak_milik_jumlah,int64,0,0.0,5826,"[1158, 323]"
